# RCPS on a small LoDoPaB-derived subset

This Colab notebook stress-tests **risk-controlling prediction sets (RCPS)** for CT image reconstruction. It trains a small CNN, uses a separate calibration split to choose and check an interval radius, and evaluates on untouched test images.

The subset provides real CT image targets. The notebook simulates 50-view noisy sinograms and reconstructs inputs with filtered back projection (FBP), so these are not the official LoDoPaB measurement files. The mirror does not document patient IDs; treat the result as an image-level exploratory experiment.

The RCPS loss is one if **any pixel** in an image falls outside its interval, and zero if all pixels are covered. The RCPS section uses 150 calibration images to select a radius and 150 separate calibration images to compute a Hoeffding upper bound on miss risk. In Colab, choose Runtime → Change runtime type → GPU, then run all cells.


In [ ]:
%pip -q install huggingface_hub scikit-image
import math
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from skimage.transform import radon, iradon, resize
from huggingface_hub import snapshot_download

SEED, SIZE = 21, 128
N_TRAIN, N_CAL, N_TEST = 700, 300, 100
N_ANGLES, NOISE_SNR_DB = 50, 30.0
ALPHAS = [0.01, 0.05, 0.10, 0.20]
EPOCHS, BATCH_SIZE = 5, 32
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
rng = np.random.default_rng(SEED); torch.manual_seed(SEED)
print('Device:', device)

In [ ]:
# Download only train/test data folders; skip the optional OOD images.
DATA_DIR = Path(snapshot_download(
    repo_id='AmirEhsan1995/lodopab-ct-glimpse', repo_type='dataset',
    allow_patterns=['train/**', 'test/**']))

def files_for(split):
    paths = sorted(p for p in DATA_DIR.rglob('*.npy') if split in p.parts)
    if not paths:
        raise FileNotFoundError(f'No .npy images for {split} under {DATA_DIR}; inspect mirror layout.')
    return paths

train_files, test_files = files_for('train'), files_for('test')
print('Images:', len(train_files), 'train,', len(test_files), 'test')
assert len(train_files) >= N_TRAIN + N_CAL and len(test_files) >= N_TEST
order = rng.permutation(len(train_files))
fit_files = [train_files[i] for i in order[:N_TRAIN]]
cal_files = [train_files[i] for i in order[N_TRAIN:N_TRAIN + N_CAL]]
test_order = rng.permutation(len(test_files))
heldout_files = [test_files[i] for i in test_order[:N_TEST]]

def load_targets(paths):
    out = []
    for p in paths:
        im = np.asarray(np.load(p), dtype=np.float32).squeeze()
        if im.ndim != 2: raise ValueError(f'Expected 2-D image, got {im.shape}: {p}')
        im = resize(im, (SIZE, SIZE), order=1, preserve_range=True, anti_aliasing=True)
        out.append(np.clip(im, 0, 1).astype(np.float32))
    return np.stack(out)

Y_fit, Y_cal, Y_test = map(load_targets, [fit_files, cal_files, heldout_files])
print('Targets:', Y_fit.shape, Y_cal.shape, Y_test.shape)

In [ ]:
# Simulate sparse-view measurements and reconstruct with filtered back projection.
angles = np.linspace(0, 180, N_ANGLES, endpoint=False)
def make_fbp_inputs(targets, seed):
    local_rng = np.random.default_rng(seed); result = []
    for i, im in enumerate(targets):
        sino = radon(im, theta=angles, circle=False).astype(np.float32)
        std = (10 ** (-NOISE_SNR_DB / 20)) * np.linalg.norm(sino) / np.sqrt(sino.size)
        noisy = sino + local_rng.normal(0, std, sino.shape).astype(np.float32)
        result.append(iradon(noisy, theta=angles, circle=False, output_size=SIZE).astype(np.float32))
        if (i + 1) % 100 == 0: print(f'FBP {i + 1}/{len(targets)}')
    return np.stack(result)
X_fit = make_fbp_inputs(Y_fit, SEED + 1)
X_cal = make_fbp_inputs(Y_cal, SEED + 2)
X_test = make_fbp_inputs(Y_test, SEED + 3)
fig, ax = plt.subplots(1, 2, figsize=(7, 3))
ax[0].imshow(X_fit[0], cmap='gray'); ax[0].set_title('Noisy sparse-view FBP')
ax[1].imshow(Y_fit[0], cmap='gray', vmin=0, vmax=1); ax[1].set_title('LoDoPaB-derived target')
for a in ax: a.axis('off')
plt.tight_layout()

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, width=32):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(1,width,3,padding=1), nn.ReLU(),
            nn.Conv2d(width,width,3,padding=1), nn.ReLU(),
            nn.Conv2d(width,width,3,padding=1), nn.ReLU(),
            nn.Conv2d(width,1,3,padding=1))
    def forward(self, x): return x + self.net(x)

def tensor4(a): return torch.from_numpy(a).float().unsqueeze(1)
loader = DataLoader(TensorDataset(tensor4(X_fit), tensor4(Y_fit)),
                    batch_size=BATCH_SIZE, shuffle=True)
model = SmallCNN().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(EPOCHS):
    model.train(); total = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        loss = F.mse_loss(model(x), y)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        total += loss.item() * len(x)
    print(f'Epoch {epoch+1}/{EPOCHS}: train MSE={total/len(loader.dataset):.6f}')

@torch.no_grad()
def predict(a):
    model.eval(); out=[]; x=tensor4(a)
    for i in range(0, len(x), BATCH_SIZE): out.append(model(x[i:i+BATCH_SIZE].to(device)).cpu())
    return torch.cat(out)
pred_cal, pred_test = predict(X_cal), predict(X_test)
# Clip centers to the known target range so the residual score exactly
# matches the intervals constructed below.
pred_cal = pred_cal.clamp(0, 1)
pred_test = pred_test.clamp(0, 1)

In [ ]:
# One maximum-pixel residual score per calibration image.
scores = (tensor4(Y_cal) - pred_cal).abs().flatten(1).amax(1).numpy()

# Scalar RCPS stress test. Use half of the calibration images to select a
# radius, and the other half once to certify its image-level miss risk.
# This is deliberately separate from split conformal above.
EPSILONS = [0.01, 0.05, 0.10, 0.20]
DELTA = 0.10
rcps_order = rng.permutation(len(scores))
n_opt = len(scores) // 2
scores_opt = scores[rcps_order[:n_opt]]
scores_risk = scores[rcps_order[n_opt:]]
n_risk = len(scores_risk)
# Fixed grid and Hoeffding margin are chosen without looking at risk-split labels.
LAMBDA_GRID = np.linspace(0.0, 2.0, 201)
hoeffding_margin = math.sqrt(math.log(1 / DELTA) / (2 * n_risk))
print(f'RCPS: n_opt={n_opt}, n_risk={n_risk}, delta={DELTA}, Hoeffding margin={hoeffding_margin:.4f}')
print('epsilon | selected radius | risk-split empirical | risk upper bound | test coverage | test mean width | status')
rcps_results = {}
for epsilon in EPSILONS:
    # Pick the narrowest radius whose optimization-split empirical risk leaves
    # room for the fixed Hoeffding margin. Selection does not use scores_risk.
    candidates = [lam for lam in LAMBDA_GRID
                  if np.mean(scores_opt > lam) <= max(0.0, epsilon - hoeffding_margin)]
    if not candidates:
        print(f'{epsilon:.2f} | -- | -- | >= {hoeffding_margin:.4f} | -- | -- | not certifiable with this bound/sample size')
        continue
    lam = float(candidates[0])
    risk_split_losses = (scores_risk > lam).astype(float)
    risk_hat = float(risk_split_losses.mean())
    risk_ucb = min(1.0, risk_hat + hoeffding_margin)
    certified = risk_ucb <= epsilon
    lo_rcps = torch.clamp(pred_test - lam, 0, 1)
    hi_rcps = torch.clamp(pred_test + lam, 0, 1)
    test_inside = (tensor4(Y_test) >= lo_rcps) & (tensor4(Y_test) <= hi_rcps)
    test_coverage = test_inside.flatten(1).all(1).float().mean().item()
    test_width = (hi_rcps - lo_rcps).mean().item()
    rcps_results[epsilon] = {'lambda': lam, 'risk_hat': risk_hat, 'risk_ucb': risk_ucb,
                           'certified': certified, 'lower': lo_rcps, 'upper': hi_rcps,
                           'test_coverage': test_coverage, 'test_width': test_width}
    status = 'certified' if certified else 'not certified'
    print(f'{epsilon:.2f} | {lam:.3f} | {risk_hat:.3f} | {risk_ucb:.3f} | {test_coverage:.3f} | {test_width:.4f} | {status}')

# At epsilon=0.10, compare the RCPS and split-conformal radii on the same test image.
epsilon_vis = 0.10
if epsilon_vis in rcps_results:
    rr = rcps_results[epsilon_vis]
    j = 0
    rcps_miss = (Y_test[j] < rr['lower'][j, 0].numpy()) | (Y_test[j] > rr['upper'][j, 0].numpy())
    print(f'Example test image: RCPS covered={not rcps_miss.any()}')
    fig, ax = plt.subplots(1, 3, figsize=(10, 3))
    ax[0].imshow(np.clip(pred_test[j, 0].numpy(), 0, 1), cmap='gray', vmin=0, vmax=1); ax[0].set_title('Prediction')
    ax[1].imshow((rr['upper'][j, 0] - rr['lower'][j, 0]).numpy(), cmap='magma'); ax[1].set_title(f'RCPS width (eps={epsilon_vis})')
    ax[2].imshow(rcps_miss, cmap='Reds'); ax[2].set_title('RCPS missed pixels')
    for a in ax: a.axis('off')
    plt.suptitle(f'RCPS radius={rr["lambda"]:.3f}; certified={rr["certified"]}'); plt.tight_layout()

## Reading the RCPS results

For each target `epsilon`, the output reports the selected interval radius, the observed miss rate on the separate risk-check calibration images, its one-sided Hoeffding upper bound, test coverage, and mean interval width. The result is marked **certified** only when the risk upper bound is no larger than `epsilon`. Test coverage is an empirical diagnostic; it is not used to select or certify the radius.

Here, `epsilon` is the allowed population probability that a new image has at least one missed pixel. `delta = 0.10` sets the confidence level of the upper bound to 90%. These describe different probabilities: epsilon is miss risk over images; confidence is the reliability of the bound across repeated risk-check calibration samples.

With 150 risk-check images, the Hoeffding margin is about 0.088. Therefore, targets `epsilon = 0.01` or `0.05` cannot be certified by this bound and sample size, even if the observed risk-check miss rate is zero. “Not certified” means the data and bound did not establish the target; it does not prove the true risk is above the target.

## Sampling assumptions and dataset limits

The Hoeffding calculation treats per-image losses as independent, identically distributed bounded observations from the same population as future images. Pixels within an image do not need to be independent because they are combined into one image-level loss. Dependence between images, such as multiple images from the same patient, could violate the assumption. The subset mirror does not document patient IDs, and its inputs are simulated rather than official LoDoPaB measurements.

## Run in Colab

Open this notebook, select a GPU runtime if available, and run the cells from top to bottom. The first cells install dependencies and download only the small train/test subset.
